Aplicamos dos tests formales de las predicciones del CAPM. El **test de Blume y Friend** (1973) es un test cruzado de dos etapas que evalúa si los betas son el único determinante del retorno esperado. El **test GRS** (Gibbons, Ross & Shanken, 1989) evalúa si los alfas de todas las acciones son conjuntamente cero.

In [1]:
#| label: setup-tests
#| code-fold: true
#| code-summary: "Datos, betas y configuración"

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import yfinance as yf
from scipy import stats
import statsmodels.api as sm

plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11})

TICKERS = ['AAPL','MSFT','AMZN','GOOGL','META','JPM','BAC','GS','WFC','MS',
           'JNJ','PFE','UNH','MRK','ABBV','XOM','CVX','KO','PG','WMT']
RF_MONTHLY = 0.0448 / 12

raw = yf.download(TICKERS, start='2015-01-01', end='2024-12-31',
                  interval='1mo', auto_adjust=True, progress=False)['Close']
returns = raw.dropna(axis=1, thresh=int(0.9*len(raw))).pct_change().dropna()
tickers_avail = returns.columns.tolist()

N = returns.shape[1]
T = len(returns)
iota = np.ones(N)

w_mkt = np.ones(N) / N
r_mkt = (returns * w_mkt).sum(axis=1)
var_mkt = r_mkt.var()

# Betas (primera etapa)
betas, alphas, avg_excess = [], [], []
for i in range(N):
    r_j = returns.iloc[:, i]
    b   = r_j.cov(r_mkt) / var_mkt
    z_j = (r_j - RF_MONTHLY).mean()
    z_m = (r_mkt - RF_MONTHLY).mean()
    betas.append(b)
    alphas.append(z_j - b * z_m)
    avg_excess.append(z_j)

betas = np.array(betas)
alphas = np.array(alphas)
avg_excess = np.array(avg_excess)
z_mkt_mean = (r_mkt - RF_MONTHLY).mean()

print(f"N={N} activos | T={T} períodos | z̄_m = {z_mkt_mean:.5f}")

N=20 activos | T=119 períodos | z̄_m = 0.01038


### Test de Blume y Friend (1973)

**Etapa 1:** Para cada acción, regresamos rendimientos en exceso contra los del mercado:
$$z_{j,t} = \alpha_j + \beta_j z_{m,t} + \varepsilon_{j,t}$$

**Etapa 2:** Regresamos los rendimientos en exceso promedio de cada acción contra los betas estimados:
$$\bar{z}_j = a + b\hat{\beta}_j + u_j$$

**Predicciones del CAPM de Sharpe-Lintner:** $a = 0$, $b = \bar{z}_m$ (prima de riesgo del mercado).  
**Predicción del CAPM de Black:** $a > 0$, $b < \bar{z}_m$ (existe una cartera de beta cero con retorno positivo).

In [2]:
#| label: blume-friend

# Segunda etapa: OLS cross-sectional
X2 = sm.add_constant(betas)
model2 = sm.OLS(avg_excess, X2).fit(cov_type='HC3')

a_hat = model2.params[0]
b_hat = model2.params[1]

print("=" * 55)
print("Test de Blume y Friend — Segunda etapa (cross-sectional)")
print("=" * 55)
print(model2.summary2().tables[1].to_string())
print()
print(f"Prima de riesgo real del mercado (z̄_m): {z_mkt_mean:.5f}")
print()
print("Comparación con predicciones teóricas:")
print(f"  Intercepto a = {a_hat:.5f}  (CAPM Sharpe predice a = 0)")
print(f"  Pendiente  b = {b_hat:.5f}  (CAPM Sharpe predice b = {z_mkt_mean:.5f})")
print()
if a_hat > 0 and model2.pvalues[0] < 0.05:
    print("→ a es significativamente positivo: consistente con CAPM de Black.")
    print("  Existe evidencia de una cartera de beta cero con retorno positivo.")
elif model2.pvalues[0] >= 0.05:
    print("→ a no es significativo: no se rechaza CAPM de Sharpe-Lintner.")
else:
    print("→ Resultado mixto.")

Test de Blume y Friend — Segunda etapa (cross-sectional)
          Coef.  Std.Err.         z     P>|z|    [0.025    0.975]
const  0.004733  0.002939  1.610379  0.107315 -0.001027  0.010493
x1     0.005650  0.002730  2.069497  0.038499  0.000299  0.011001

Prima de riesgo real del mercado (z̄_m): 0.01038

Comparación con predicciones teóricas:
  Intercepto a = 0.00473  (CAPM Sharpe predice a = 0)
  Pendiente  b = 0.00565  (CAPM Sharpe predice b = 0.01038)

→ a no es significativo: no se rechaza CAPM de Sharpe-Lintner.


In [3]:
#| label: bf-plot
#| code-fold: true
#| fig-cap: "Test de Blume y Friend: retornos en exceso promedio vs betas estimados"

beta_range = np.linspace(betas.min() - 0.1, betas.max() + 0.1, 100)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(betas, avg_excess * 100, color='#2563eb', s=70, zorder=5, label='Acciones')
for i, t in enumerate(tickers_avail):
    ax.annotate(t, (betas[i], avg_excess[i]*100),
                fontsize=7.5, xytext=(3, 2), textcoords='offset points')

# Regresión estimada
ax.plot(beta_range, (a_hat + b_hat * beta_range) * 100,
        color='#dc2626', linewidth=2, label=f'Regresión: a={a_hat:.4f}, b={b_hat:.4f}')
# SML teórica
ax.plot(beta_range, z_mkt_mean * beta_range * 100,
        color='#16a34a', linewidth=2, linestyle='--',
        label=f'SML teórica (a=0, b={z_mkt_mean:.4f})')

ax.set_xlabel('Beta estimado (β̂)', fontsize=12)
ax.set_ylabel('Retorno en exceso promedio (%)', fontsize=12)
ax.set_title('Test de Blume y Friend — Segunda etapa', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### Test GRS (Gibbons, Ross & Shanken, 1989)

El test GRS evalúa si los alfas de **todas** las acciones son conjuntamente cero. Es un test F de restricciones lineales sobre un sistema SUR de regresiones.

$$J_1 = \frac{T-N-1}{N} \cdot \frac{SR^2_q - SR^2_m}{1 + SR^2_m} \sim F_{N,\, T-N-1}$$

Donde $SR_q$ es la razón de Sharpe de la cartera tangente y $SR_m$ la del portafolio de mercado.

**$H_0$:** $\alpha_j = 0 \ \forall j$ (el portafolio de mercado es eficiente en media-varianza).

In [4]:
#| label: grs-test

from scipy.stats import f as f_dist

SR_m = (r_mkt.mean() - RF_MONTHLY) / r_mkt.std()

# Cartera tangente (máximo Sharpe)
V_mat = returns.cov().values
V_inv = np.linalg.inv(V_mat)
excess_vec = returns.mean().values - RF_MONTHLY
z_q = V_inv @ excess_vec
w_q = z_q / (iota @ z_q)
r_q_series = (returns.values @ w_q)
SR_q = (r_q_series.mean() - RF_MONTHLY) / r_q_series.std()

J1 = ((T - N - 1) / N) * (SR_q**2 - SR_m**2) / (1 + SR_m**2)
p_grs = 1 - f_dist.cdf(J1, dfn=N, dfd=T - N - 1)
f_critical = f_dist.ppf(0.95, dfn=N, dfd=T - N - 1)

print("=" * 50)
print("Test GRS (Gibbons, Ross & Shanken, 1989)")
print("=" * 50)
print(f"N = {N} acciones  |  T = {T} períodos")
print(f"Grados de libertad: F({N}, {T-N-1})")
print()
print(f"SR mercado (SR_m):    {SR_m:.4f}")
print(f"SR tangente (SR_q):   {SR_q:.4f}")
print()
print(f"Estadístico J1:       {J1:.4f}")
print(f"Valor crítico (5%):   {f_critical:.4f}")
print(f"p-value:              {p_grs:.4f}")
print()
if J1 > f_critical:
    print("→ Se rechaza H₀: el portafolio de mercado NO es eficiente en media-varianza.")
    print("  Los alfas son conjuntamente diferentes de cero.")
else:
    print("→ No se rechaza H₀: no hay evidencia suficiente contra la eficiencia del mercado.")

Test GRS (Gibbons, Ross & Shanken, 1989)
N = 20 acciones  |  T = 119 períodos
Grados de libertad: F(20, 98)

SR mercado (SR_m):    0.2416
SR tangente (SR_q):   0.5138

Estadístico J1:       0.9520
Valor crítico (5%):   1.6786
p-value:              0.5252

→ No se rechaza H₀: no hay evidencia suficiente contra la eficiencia del mercado.
